# LLM 기초: 밑바닥부터 만드는 대규모 언어모델
### Google Colab 실습 노트북

이 노트북은 책 **《LLM 기초: 밑바닥부터 만드는 대규모 언어모델》**(Highmaru Press, dirmich 지음)의 모든 코드를 순서대로 실행할 수 있도록 정리한 것입니다.

- 상단 메뉴 `런타임 > 런타임 유형 변경`에서 GPU(T4 이상)를 선택하고 시작하세요.
- 셀을 위에서부터 순서대로 `Shift+Enter`로 실행하세요.
- 9~11장은 별도로 transformers, datasets, peft 패키지를 추가로 다운로드합니다.

# 서문: 이 책을 읽는 방법

## 이 책의 목표

이 책은 ChatGPT, GPT-4 같은 대규모 언어모델(LLM, Large Language Model)이 **내부적으로 어떻게 동작하는지**를 처음부터 끝까지 코드로 직접 구현하며 배우는 실습 교재입니다.

이론만 배우면 "어텐션이 중요하다"는 말은 이해해도 실제로 왜, 어떻게 동작하는지 감이 잡히지 않습니다. 반대로 코드만 따라 치면 원리를 모른 채 복사-붙여넣기만 하게 됩니다. 이 책은 두 가지를 함께 가져갑니다.

- **이론**: 각 장의 앞부분에서 개념을 그림과 수식 없이도 이해할 수 있도록 비유와 함께 설명합니다.
- **실습**: 모든 장은 Google Colab에서 그대로 실행 가능한 파이썬 코드를 포함합니다. PyTorch를 사용하여 토크나이저, 임베딩, 어텐션, 트랜스포머 블록을 한 줄씩 직접 만들고, 최종적으로 작은 GPT 모델을 학습시켜 텍스트를 생성합니다.

이 책을 끝까지 따라가면 다음을 직접 구현하게 됩니다.

1. 텍스트를 토큰으로 쪼개는 BPE 토크나이저
2. 단어를 벡터로 바꾸는 임베딩 층
3. Self-Attention과 Multi-Head Attention
4. 트랜스포머 블록 (LayerNorm, FeedForward, Residual Connection)
5. 미니 GPT 모델 전체 구조
6. 학습 루프와 손실 함수
7. 텍스트 생성 샘플링 전략 (top-k, top-p, temperature)
8. Hugging Face의 사전학습 모델 불러오기
9. LoRA를 이용한 가벼운 파인튜닝

## 필요한 사전 지식

파이썬 문법을 알고, 딥러닝을 처음 접하더라도 괜찮습니다. 다만 아래 개념을 들어본 적이 있으면 훨씬 수월합니다.

- 행렬 곱셈이 무엇인지 (내적, 차원)
- 신경망의 기본 구조 (입력층, 은닉층, 출력층)
- 경사하강법(gradient descent)이라는 말을 들어본 적

모르더라도 각 장에서 필요한 만큼 짧게 설명하고 넘어갑니다.

## Google Colab 사용법

Google Colab은 브라우저에서 무료로 GPU를 사용할 수 있는 파이썬 실행 환경입니다. 별도 설치 없이 아래 순서로 시작하세요.

1. [colab.research.google.com](https://colab.research.google.com) 접속 후 로그인
2. 상단 메뉴에서 `파일 > 새 노트` 클릭
3. `런타임 > 런타임 유형 변경`에서 하드웨어 가속기를 **GPU** (T4 이상 권장)로 설정
4. 각 코드 블록을 순서대로 하나의 셀에 붙여넣고 `Shift + Enter`로 실행

책에서 코드 블록이 나올 때마다 주석으로 다음과 같이 표시합니다.

이 주석이 나오면 새로운 셀에 붙여넣고 실행한다고 생각하면 됩니다. 같은 장 안에서 이어지는 셀들은 이전 셀에서 정의한 변수와 함수를 계속 사용하므로, **반드시 순서대로 실행**해야 합니다.

## 모듈 구조 안내

실습 코드는 하나의 거대한 스크립트가 아니라, 실제 오픈소스 LLM 라이브러리들처럼 역할별로 나눈 모듈로 구성합니다.

```
mini_gpt/
├── tokenizer.py       # 2장: BPE 토크나이저
├── embedding.py        # 3장: 토큰 임베딩 + 위치 인코딩
├── attention.py         # 4장: Self-Attention, Multi-Head Attention
├── transformer_block.py # 5장: 트랜스포머 블록
├── model.py             # 6장: 전체 GPT 모델 조립
├── train.py              # 7장: 학습 루프
└── generate.py           # 8장: 텍스트 생성
```

Colab에서는 파일을 여러 개 만드는 대신 `%%writefile` 매직 명령으로 각 모듈을 파일로 저장하고, 이후 셀에서 `import`해서 사용합니다. 이렇게 하면 실제 프로젝트처럼 모듈화된 구조를 Colab 환경에서도 그대로 체험할 수 있습니다.

그럼 시작해봅시다.

# LLM이란 무엇인가

## 언어모델의 정의

언어모델(Language Model)은 아주 단순하게 말하면 **"다음에 올 단어를 확률적으로 맞히는 기계"**입니다.

> "오늘 저녁에 ___을 먹었다"

빈칸에 "밥", "치킨", "라면" 같은 단어는 확률이 높고, "자동차", "우주선" 같은 단어는 확률이 낮습니다. 언어모델은 방대한 텍스트를 학습하면서 이런 확률 분포를 통계적으로 익힙니다. ChatGPT 같은 모델이 문장을 만들어내는 과정도 근본적으로는 "다음 토큰을 하나씩 예측해서 이어붙이는" 반복 작업입니다.

이때 "대규모(Large)"라는 말이 붙는 이유는 두 가지입니다.

1. **파라미터 수가 크다**: 수억~수조 개의 학습 가능한 가중치를 가집니다.
2. **학습 데이터가 크다**: 인터넷 텍스트, 책, 코드 등 수천억~수조 토큰을 학습합니다.

## 왜 트랜스포머인가

2017년 논문 "Attention Is All You Need"에서 제안된 **트랜스포머(Transformer)** 구조는 현재 거의 모든 LLM의 기반입니다. 트랜스포머 이전에는 RNN(순환신경망)이 주로 쓰였는데, RNN은 문장을 한 단어씩 순서대로 처리해야 해서 느리고, 문장이 길어지면 앞쪽 정보를 잊어버리는 문제가 있었습니다.

트랜스포머는 **어텐션(Attention)** 메커니즘을 이용해 문장의 모든 단어를 동시에 보면서, 각 단어가 다른 단어들과 얼마나 관련 있는지를 계산합니다. 이렇게 하면:

- 병렬 처리가 가능해 학습이 빠르다
- 문장이 길어도 먼 거리의 단어 관계를 잘 포착한다

이 책에서 만들 미니 GPT도 이 트랜스포머 구조를 그대로 축소한 버전입니다.

## GPT 계열 모델의 동작 원리 한눈에 보기

GPT(Generative Pre-trained Transformer) 계열 모델은 다음 파이프라인으로 동작합니다.

```
텍스트 입력
   │
   ▼
[토크나이저] 텍스트 → 토큰 ID 시퀀스
   │
   ▼
[임베딩] 토큰 ID → 벡터, 위치 정보 추가
   │
   ▼
[트랜스포머 블록 × N] Self-Attention + FeedForward 반복
   │
   ▼
[출력층] 다음 토큰의 확률 분포 계산
   │
   ▼
[샘플링] 확률 분포에서 다음 토큰 선택
   │
   ▼
생성된 토큰을 입력에 이어붙이고 반복
```

이 책의 2장부터 8장까지가 정확히 이 파이프라인의 각 단계에 대응합니다. 즉 이 다이어그램이 이 책 전체의 지도입니다.

## 사전학습과 파인튜닝

LLM은 보통 두 단계를 거쳐 만들어집니다.

- **사전학습(Pre-training)**: 대량의 텍스트로 "다음 토큰 맞히기"만 반복 학습합니다. 이 단계가 끝나면 문법, 상식, 세계 지식을 어느 정도 갖춘 모델이 됩니다. 하지만 아직 "질문에 답하는 방식"을 모릅니다.
- **파인튜닝(Fine-tuning)**: 사전학습된 모델을 특정 목적(대화, 지시 수행, 코드 작성 등)에 맞게 소량의 데이터로 추가 학습합니다. 여기에는 지도학습 파인튜닝(SFT), 인간 피드백 강화학습(RLHF), LoRA 같은 경량화 기법 등이 있습니다.

이 책은 6~7장에서 미니 GPT를 처음부터 사전학습시켜보고, 11장에서 사전학습된 실제 모델을 LoRA로 가볍게 파인튜닝하는 것까지 다룹니다.

## 개발 환경 준비 (첫 Colab 셀)

이제 첫 코드를 실행해봅시다. Colab에서 GPU가 잘 잡히는지 확인하고, 필요한 패키지를 설치합니다.

In [ ]:
# GPU 확인
import torch
print("PyTorch 버전:", torch.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU 이름:", torch.cuda.get_device_name(0))

device = "cuda" if torch.cuda.is_available() else "cpu"
print("사용할 디바이스:", device)

In [ ]:
# 이 책 전체에서 사용할 패키지 설치
!pip install -q torch transformers datasets tiktoken regex peft accelerate

`torch.cuda.is_available()`이 `False`로 나온다면 `런타임 > 런타임 유형 변경`에서 GPU를 T4로 설정했는지 확인하세요. CPU만으로도 이 책의 실습은 진행할 수 있지만, 7장 학습 단계는 GPU가 훨씬 빠릅니다.

다음 장부터는 실제로 텍스트를 모델이 이해할 수 있는 숫자로 바꾸는 **토크나이저**를 직접 만들어보겠습니다.

# 토크나이제이션: 텍스트를 숫자로

## 왜 토큰이 필요한가

신경망은 숫자만 이해합니다. "안녕하세요"라는 문자열을 모델에 그대로 넣을 수는 없고, 정수 ID의 나열로 바꿔야 합니다. 이 변환 과정을 **토크나이제이션(Tokenization)**이라고 하고, 변환 규칙을 담은 도구를 **토크나이저(Tokenizer)**라고 합니다.

가장 단순한 방법은 "단어 단위로 쪼개서 각 단어에 번호를 붙이기"입니다. 하지만 이 방법은 두 가지 문제가 있습니다.

1. 사전에 없는 단어(신조어, 오타, 외국어)를 처리할 수 없다.
2. 단어 수가 너무 많아지면 어휘 사전(vocabulary)이 지나치게 커진다.

반대로 "글자 단위로 쪼개기"는 어휘 사전은 작아지지만, 문장이 지나치게 길어지고 글자 하나하나에는 의미가 거의 없어 학습이 비효율적입니다.

## BPE: 두 방식의 절충안

**BPE(Byte Pair Encoding)**는 실제 GPT 계열 모델 대부분이 사용하는 방식으로, 자주 등장하는 글자 쌍을 반복적으로 합쳐가며 어휘 사전을 만듭니다. 예를 들어 "low", "lower", "lowest"라는 단어들이 자주 나온다면, `l`+`o` → `lo`, `lo`+`w` → `low` 처럼 자주 붙어 다니는 조각들을 점점 큰 단위로 합칩니다.

결과적으로 자주 쓰이는 단어는 통째로 하나의 토큰이 되고, 드문 단어는 여러 개의 서브워드(subword) 토큰으로 쪼개집니다. 이렇게 하면 어휘 사전 크기를 조절할 수 있으면서도 처음 보는 단어도 서브워드 조합으로 표현할 수 있습니다.

## BPE 학습 알고리즘

1. 텍스트를 글자 단위로 쪼갠다.
2. 가장 자주 붙어서 등장하는 인접한 두 토큰 쌍을 찾는다.
3. 그 쌍을 하나의 새 토큰으로 합친다.
4. 원하는 어휘 사전 크기에 도달할 때까지 2~3을 반복한다.

이제 직접 구현해봅니다. Colab에서 새 셀에 아래 코드를 순서대로 입력하세요.

In [ ]:
# mini_gpt 패키지 폴더 생성
import os
os.makedirs("mini_gpt", exist_ok=True)

In [ ]:
%%writefile mini_gpt/tokenizer.py
"""BPE(Byte Pair Encoding) 토크나이저 - 처음부터 구현"""
from collections import Counter, defaultdict


class BPETokenizer:
    def __init__(self):
        self.merges = {}          # (토큰1, 토큰2) -> 합쳐진 새 토큰 ID
        self.vocab = {}           # 토큰 ID -> 바이트 시퀀스
        self.token_to_id = {}     # 문자열 -> 토큰 ID

    def _get_pair_counts(self, token_ids_list):
        """모든 시퀀스에서 인접한 토큰 쌍의 등장 횟수를 센다."""
        counts = Counter()
        for ids in token_ids_list:
            for a, b in zip(ids, ids[1:]):
                counts[(a, b)] += 1
        return counts

    def _merge(self, ids, pair, new_id):
        """ids 안에서 pair를 new_id로 모두 합친다."""
        merged = []
        i = 0
        while i < len(ids):
            if i < len(ids) - 1 and (ids[i], ids[i + 1]) == pair:
                merged.append(new_id)
                i += 2
            else:
                merged.append(ids[i])
                i += 1
        return merged

    def train(self, text, vocab_size=500):
        """텍스트로부터 BPE 어휘 사전을 학습한다."""
        # 1단계: 초기 어휘는 UTF-8 바이트 256개
        for i in range(256):
            self.vocab[i] = bytes([i])

        # 텍스트를 바이트 단위 정수 리스트로 변환
        ids = list(text.encode("utf-8"))
        token_ids_list = [ids]  # 문서가 하나뿐이라 리스트 안에 하나만

        num_merges = vocab_size - 256
        for step in range(num_merges):
            pair_counts = self._get_pair_counts(token_ids_list)
            if not pair_counts:
                break
            best_pair = max(pair_counts, key=pair_counts.get)
            new_id = 256 + step
            self.vocab[new_id] = self.vocab[best_pair[0]] + self.vocab[best_pair[1]]
            self.merges[best_pair] = new_id
            token_ids_list = [self._merge(ids, best_pair, new_id) for ids in token_ids_list]

        return self

    def encode(self, text):
        """텍스트를 토큰 ID 리스트로 변환한다."""
        ids = list(text.encode("utf-8"))
        while len(ids) >= 2:
            pair_counts = self._get_pair_counts([ids])
            # 학습 때 만들어진 병합 중, 등장한 쌍 가운데 가장 먼저 만들어진 병합을 우선 적용
            candidates = [p for p in pair_counts if p in self.merges]
            if not candidates:
                break
            best_pair = min(candidates, key=lambda p: self.merges[p])
            ids = self._merge(ids, best_pair, self.merges[best_pair])
        return ids

    def decode(self, ids):
        """토큰 ID 리스트를 다시 텍스트로 복원한다."""
        byte_seq = b"".join(self.vocab[i] for i in ids)
        return byte_seq.decode("utf-8", errors="replace")

이제 이 토크나이저를 학습시키고 실제로 인코딩/디코딩을 해봅니다.

In [ ]:
from mini_gpt.tokenizer import BPETokenizer

sample_text = """
인공지능은 인간의 학습능력과 추론능력, 지각능력을 인공적으로 구현하려는 컴퓨터 과학의 세부 분야이다.
대규모 언어모델은 방대한 텍스트 데이터를 학습하여 다음 단어를 예측하는 방식으로 언어를 익힌다.
트랜스포머 구조는 어텐션 메커니즘을 핵심으로 사용한다.
""" * 20  # 반복해서 병합 패턴이 잘 드러나도록 함

tokenizer = BPETokenizer()
tokenizer.train(sample_text, vocab_size=500)

test = "트랜스포머는 어텐션을 사용한다."
encoded = tokenizer.encode(test)
decoded = tokenizer.decode(encoded)

print("원문:", test)
print("토큰 ID 개수:", len(encoded))
print("토큰 ID:", encoded)
print("복원된 텍스트:", decoded)
print("원문과 동일한가?", test == decoded)

바이트 단위로 시작하기 때문에 한글, 영어, 이모지, 특수문자 등 어떤 텍스트가 들어와도 처리할 수 있다는 점이 이 방식의 큰 장점입니다. 반복 실행해보면 자주 등장한 한글 조각(예: "언어모델", "트랜스포머")이 하나의 토큰으로 합쳐진 것을 `tokenizer.vocab`에서 확인할 수 있습니다.

## 실전에서는 tiktoken을 사용합니다

우리가 만든 토크나이저는 원리를 이해하기 위한 교육용입니다. 실제로는 학습 속도가 훨씬 빠르고 이미 대규모 데이터로 어휘 사전이 구축된 라이브러리를 사용합니다. OpenAI가 공개한 `tiktoken`이 대표적입니다.

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("gpt2")
test = "Transformers use attention mechanisms."
ids = enc.encode(test)
print("토큰 ID:", ids)
print("토큰 개수:", len(ids))
print("복원:", enc.decode(ids))

# 토큰 단위로 잘려나간 조각 확인
for tid in ids:
    print(tid, "->", repr(enc.decode([tid])))

이후 6~7장에서 미니 GPT를 학습시킬 때는 우리가 만든 `BPETokenizer`를 그대로 사용해 원리를 유지하되, 실제 서비스에서는 `tiktoken`처럼 검증된 라이브러리를 쓴다는 점을 기억해두세요.

다음 장에서는 이렇게 얻은 토큰 ID를 모델이 의미를 담아 다룰 수 있는 **벡터**로 바꾸는 임베딩 층을 만들어보겠습니다.

# 임베딩과 위치 인코딩

## 토큰 ID만으로는 부족한 이유

토크나이저를 거치면 텍스트는 `[1023, 45, 892, ...]` 같은 정수 리스트가 됩니다. 하지만 정수 자체는 의미가 없습니다. 토큰 ID 1023과 1024가 이웃한 숫자라고 해서 두 토큰의 의미가 비슷한 것은 전혀 아닙니다.

그래서 각 토큰 ID를 **의미를 담은 실수 벡터**로 바꾸는 과정이 필요합니다. 이것이 **임베딩(Embedding)**입니다. 임베딩 벡터는 학습을 통해 점점 "의미가 비슷한 토큰은 벡터 공간에서 가까운 위치에" 놓이도록 조정됩니다. 예를 들어 학습이 잘 되면 "고양이"와 "강아지"의 임베딩 벡터는 "고양이"와 "자동차"보다 서로 더 가까워집니다.

구현 관점에서 임베딩 층은 사실 단순한 **조회 테이블(lookup table)**입니다. 어휘 사전 크기가 V, 임베딩 차원이 d라면, (V, d) 크기의 행렬을 만들고, 토큰 ID를 그 행렬의 행 번호로 사용해서 해당 벡터를 꺼내옵니다.

## 위치 정보가 따로 필요한 이유

트랜스포머의 어텐션 메커니즘은 문장 안의 모든 토큰을 동시에 병렬로 처리합니다. 이는 속도 면에서는 큰 장점이지만, 부작용으로 **"어떤 토큰이 몇 번째에 있는지"에 대한 정보가 자동으로 주어지지 않습니다.** RNN은 순서대로 처리하기 때문에 위치 정보가 자연히 반영되지만, 트랜스포머는 "나는 밥을 먹었다"와 "밥을 나는 먹었다"를 순서 정보 없이는 구분하지 못합니다.

그래서 각 위치(1번째, 2번째, 3번째...)마다 고유한 벡터를 더해주는 **위치 인코딩(Positional Encoding)**을 사용합니다. 방식은 여러 가지가 있습니다.

- **사인/코사인 함수 기반**(원조 트랜스포머 논문): 위치마다 서로 다른 주파수의 사인·코사인 값을 계산해서 사용. 학습 파라미터가 없음.
- **학습형 위치 임베딩**(GPT-2 등): 위치 인덱스도 토큰처럼 하나의 임베딩 테이블로 학습.

이 책에서는 GPT-2와 동일한 방식인 학습형 위치 임베딩을 사용합니다. 구현이 더 간단하고 이해하기 쉽습니다.

## 구현하기

In [ ]:
%%writefile mini_gpt/embedding.py
"""토큰 임베딩 + 위치 임베딩"""
import torch
import torch.nn as nn


class GPTEmbedding(nn.Module):
    def __init__(self, vocab_size, embed_dim, max_seq_len, dropout=0.1):
        super().__init__()
        # 토큰 ID -> 벡터 (조회 테이블), 크기: (vocab_size, embed_dim)
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        # 위치 인덱스 -> 벡터, 크기: (max_seq_len, embed_dim)
        self.position_embedding = nn.Embedding(max_seq_len, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, token_ids):
        # token_ids: (batch_size, seq_len)
        batch_size, seq_len = token_ids.shape
        positions = torch.arange(seq_len, device=token_ids.device).unsqueeze(0)
        # positions: (1, seq_len) -> 브로드캐스팅으로 배치 전체에 적용

        token_vecs = self.token_embedding(token_ids)      # (batch, seq_len, embed_dim)
        position_vecs = self.position_embedding(positions)  # (1, seq_len, embed_dim)

        # 두 벡터를 더해서 "이 토큰이면서 동시에 이 위치"라는 정보를 함께 담는다
        embeddings = token_vecs + position_vecs
        return self.dropout(embeddings)

## 직접 동작 확인해보기

In [ ]:
from mini_gpt.embedding import GPTEmbedding
import torch

vocab_size = 500      # 2장에서 만든 토크나이저의 어휘 크기
embed_dim = 64         # 각 토큰을 표현할 벡터 차원
max_seq_len = 32        # 모델이 한 번에 볼 수 있는 최대 토큰 수

embedding_layer = GPTEmbedding(vocab_size, embed_dim, max_seq_len)

# 가짜 토큰 ID 배치: (batch_size=2, seq_len=5)
fake_token_ids = torch.tensor([
    [10, 25, 3, 88, 42],
    [7,  7,  7, 7,  7],
])

output = embedding_layer(fake_token_ids)
print("입력 shape:", fake_token_ids.shape)
print("출력 shape:", output.shape)   # (2, 5, 64) 이어야 함
print("첫 번째 문장, 첫 토큰의 임베딩 벡터 일부:", output[0, 0, :8])

`fake_token_ids`의 두 번째 문장은 같은 토큰 `7`이 5번 반복됩니다. 만약 위치 임베딩이 없다면 5개 위치의 출력 벡터가 완전히 동일했을 것입니다. 아래 코드로 직접 확인해보세요.

In [ ]:
# 같은 토큰이라도 위치가 다르면 임베딩 결과가 달라지는지 확인
second_sentence_output = output[1]  # (5, 64)
for i in range(5):
    print(f"위치 {i} 벡터 일부:", second_sentence_output[i, :4].tolist())

5개 벡터가 모두 다르게 출력됩니다. 이것이 바로 위치 인코딩이 "순서 정보"를 임베딩에 주입하는 방식입니다.

## 임베딩 차원(embed_dim)을 정하는 기준

`embed_dim`은 하나의 토큰을 몇 개의 실수로 표현할지를 정하는 하이퍼파라미터입니다. 값이 클수록 더 풍부한 의미를 담을 수 있지만, 계산량과 메모리 사용량도 함께 늘어납니다. 참고로 실제 모델들의 대략적인 규모는 다음과 같습니다.

| 모델 | 임베딩 차원 | 레이어 수 |
|---|---|---|
| GPT-2 Small | 768 | 12 |
| GPT-2 XL | 1600 | 48 |
| 이 책의 미니 GPT | 64~128 | 4~6 |

이 책은 Colab 무료 GPU에서도 몇 분 안에 학습이 끝나도록 매우 작은 규모로 설계했습니다. 원리는 실제 대규모 모델과 완전히 동일하며, 차원과 레이어 수만 늘리면 그대로 확장됩니다.

다음 장에서는 이 임베딩 벡터들이 서로 정보를 주고받는 핵심 메커니즘인 **어텐션(Attention)**을 구현합니다.

# 어텐션 메커니즘

## 핵심 아이디어: 문맥에 따라 의미가 달라진다

"배가 아프다"와 "배를 타고 강을 건넜다"와 "배를 사서 먹었다"에서 "배"라는 단어는 각각 신체 부위, 탈것, 과일이라는 전혀 다른 의미를 가집니다. 임베딩만으로는 "배"라는 토큰에 항상 같은 벡터가 배정되므로 이 차이를 표현할 수 없습니다.

**어텐션(Attention)**은 각 단어가 문장 속 다른 단어들을 "얼마나 참고할지" 가중치를 계산해서, 문맥에 맞게 벡터를 다시 조합해주는 메커니즘입니다. "배"라는 단어는 주변에 "아프다"가 있으면 그쪽에 더 집중(attend)하고, "타고"가 있으면 그쪽에 더 집중하도록 스스로 학습합니다.

## Query, Key, Value 비유

어텐션은 도서관에서 책을 찾는 과정에 비유할 수 있습니다.

- **Query(질의)**: "나는 지금 이런 정보가 필요해"라는 질문 벡터. 지금 처리 중인 단어가 만듭니다.
- **Key(열쇠)**: 각 단어가 "나는 이런 정보를 갖고 있어"라고 내거는 색인 벡터.
- **Value(값)**: 실제로 전달할 내용이 담긴 벡터.

Query와 모든 단어의 Key를 비교(내적)해서 유사도 점수를 계산하고, 이 점수를 확률처럼 정규화(softmax)한 뒤, 그 확률로 Value들을 가중합산합니다. 즉 "질문과 관련도가 높은 단어의 내용을 더 많이 반영해서 새 벡터를 만든다"는 것이 어텐션의 전부입니다.

수식으로 쓰면 다음과 같습니다.

```
Attention(Q, K, V) = softmax( Q · K^T / sqrt(d_k) ) · V
```

- `Q · K^T`: Query와 Key의 내적으로 유사도 점수 행렬을 계산
- `sqrt(d_k)`로 나누는 이유: 차원이 커질수록 내적 값이 커져 softmax가 한쪽으로 치우치는 것을 방지 (스케일 조정)
- `softmax`: 점수를 합이 1인 확률 분포로 변환
- 마지막에 `V`를 곱해 확률 가중 평균으로 새 벡터를 계산

## Self-Attention 구현

In [ ]:
%%writefile mini_gpt/attention.py
"""Self-Attention과 Multi-Head Attention 구현"""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class SelfAttention(nn.Module):
    """가장 기본적인 단일 헤드 Self-Attention"""

    def __init__(self, embed_dim, head_dim, max_seq_len, dropout=0.1):
        super().__init__()
        # Q, K, V를 만드는 선형 변환. 입력과 출력 차원이 다를 수 있음(head_dim)
        self.query_proj = nn.Linear(embed_dim, head_dim, bias=False)
        self.key_proj = nn.Linear(embed_dim, head_dim, bias=False)
        self.value_proj = nn.Linear(embed_dim, head_dim, bias=False)
        self.dropout = nn.Dropout(dropout)

        # 미래 토큰을 못 보게 막는 causal mask (하삼각행렬)
        # GPT는 "다음 단어 예측"이 목적이므로 미래 정보를 참조하면 반칙(치팅)이 됩니다.
        causal_mask = torch.tril(torch.ones(max_seq_len, max_seq_len))
        self.register_buffer("causal_mask", causal_mask)

    def forward(self, x):
        # x: (batch, seq_len, embed_dim)
        batch_size, seq_len, _ = x.shape

        Q = self.query_proj(x)   # (batch, seq_len, head_dim)
        K = self.key_proj(x)     # (batch, seq_len, head_dim)
        V = self.value_proj(x)   # (batch, seq_len, head_dim)

        head_dim = Q.shape[-1]
        # 유사도 점수: (batch, seq_len, seq_len)
        scores = Q @ K.transpose(-2, -1) / math.sqrt(head_dim)

        # 미래 위치는 -inf로 마스킹 -> softmax 후 확률이 0이 됨
        mask = self.causal_mask[:seq_len, :seq_len]
        scores = scores.masked_fill(mask == 0, float("-inf"))

        attn_weights = F.softmax(scores, dim=-1)  # 각 행의 합이 1
        attn_weights = self.dropout(attn_weights)

        output = attn_weights @ V  # (batch, seq_len, head_dim)
        return output, attn_weights


class MultiHeadAttention(nn.Module):
    """여러 개의 헤드가 서로 다른 관점에서 병렬로 어텐션을 계산"""

    def __init__(self, embed_dim, num_heads, max_seq_len, dropout=0.1):
        super().__init__()
        assert embed_dim % num_heads == 0, "embed_dim은 num_heads로 나누어 떨어져야 합니다"
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.qkv_proj = nn.Linear(embed_dim, embed_dim * 3, bias=False)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

        causal_mask = torch.tril(torch.ones(max_seq_len, max_seq_len))
        self.register_buffer("causal_mask", causal_mask)

    def forward(self, x):
        batch_size, seq_len, embed_dim = x.shape

        # 한번에 Q, K, V를 모두 계산 (효율을 위해 하나의 선형층으로 처리)
        qkv = self.qkv_proj(x)  # (batch, seq_len, embed_dim*3)
        Q, K, V = qkv.chunk(3, dim=-1)

        # (batch, seq_len, embed_dim) -> (batch, num_heads, seq_len, head_dim)
        def split_heads(t):
            t = t.view(batch_size, seq_len, self.num_heads, self.head_dim)
            return t.transpose(1, 2)

        Q, K, V = split_heads(Q), split_heads(K), split_heads(V)

        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.head_dim)
        mask = self.causal_mask[:seq_len, :seq_len]
        scores = scores.masked_fill(mask == 0, float("-inf"))

        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        out = attn_weights @ V  # (batch, num_heads, seq_len, head_dim)

        # 다시 (batch, seq_len, embed_dim)로 합치기
        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, embed_dim)
        return self.out_proj(out)

## 동작 확인 및 어텐션 가중치 시각화

In [ ]:
from mini_gpt.attention import SelfAttention, MultiHeadAttention
import torch

embed_dim = 32
seq_len = 6
batch_size = 1

x = torch.randn(batch_size, seq_len, embed_dim)

attn = SelfAttention(embed_dim, head_dim=embed_dim, max_seq_len=16)
output, weights = attn(x)

print("입력 shape:", x.shape)
print("출력 shape:", output.shape)
print("어텐션 가중치 shape:", weights.shape)  # (1, 6, 6)
print("\n어텐션 가중치 행렬 (행=현재 토큰, 열=참조하는 토큰):")
for row in weights[0]:
    print([round(v, 2) for v in row.tolist()])

출력된 행렬을 보면 상삼각 부분(미래 위치)이 전부 0인 것을 확인할 수 있습니다. 이것이 바로 `causal_mask`가 미래 정보를 차단한 결과입니다. 각 행의 합은 정확히 1이 되는데, softmax가 확률 분포를 만들기 때문입니다.

In [ ]:
# Multi-Head Attention도 동일하게 확인
mha = MultiHeadAttention(embed_dim=32, num_heads=4, max_seq_len=16)
out = mha(x)
print("Multi-Head Attention 출력 shape:", out.shape)  # (1, 6, 32) - 입력과 동일한 shape 유지

## 헤드를 여러 개 쓰는 이유

하나의 어텐션 헤드는 한 가지 관점에서만 단어 간 관계를 계산합니다. 예를 들어 어떤 헤드는 문법적 관계(주어-동사)에 집중하고, 다른 헤드는 의미적 유사성에 집중할 수 있습니다. **Multi-Head Attention**은 임베딩 차원을 여러 조각(head)으로 나눠서 각 조각이 독립적으로 어텐션을 계산한 뒤 다시 합칩니다. 이렇게 하면 모델이 동시에 여러 종류의 관계를 병렬로 포착할 수 있습니다.

다음 장에서는 이 어텐션을 감싸서 실제로 학습이 안정적으로 되도록 만드는 **트랜스포머 블록**(LayerNorm, FeedForward, Residual Connection)을 구현합니다.

# 트랜스포머 블록 조립하기

앞 장에서 만든 Multi-Head Attention은 그 자체만으로는 학습이 잘 되지 않습니다. 실제 트랜스포머는 어텐션 주위에 세 가지 장치를 추가로 사용합니다: **잔차 연결(Residual Connection)**, **레이어 정규화(Layer Normalization)**, **피드포워드 네트워크(FeedForward Network)**. 이 셋을 어텐션과 합친 하나의 단위를 **트랜스포머 블록**이라 부르고, 이 블록을 여러 겹 쌓아서 모델을 만듭니다.

## 잔차 연결(Residual Connection): 정보가 사라지지 않게

레이어를 깊게 쌓을수록 원래 정보가 여러 번의 변환을 거치며 희미해지거나, 학습 중 그래디언트(기울기)가 소실되는 문제가 발생합니다. 잔차 연결은 간단한 트릭으로 이를 해결합니다.

```
출력 = 어텐션(x) + x
```

즉 변환한 결과를 그대로 쓰지 않고, **입력 x를 다시 한 번 더해줍니다.** 이렇게 하면 최악의 경우에도(어텐션이 아무 도움이 안 되더라도) 최소한 입력 정보는 그대로 다음 층에 전달됩니다. 마치 "지름길"을 하나 더 만들어주는 것과 같습니다. ResNet이라는 이미지 모델에서 처음 제안되어 트랜스포머에도 그대로 쓰입니다.

## 레이어 정규화(Layer Normalization): 학습을 안정시키기

신경망이 깊어지면 각 층을 통과할 때마다 값의 크기(스케일)가 들쭉날쭉해져 학습이 불안정해집니다. LayerNorm은 각 토큰의 벡터를 평균 0, 분산 1이 되도록 정규화한 뒤, 학습 가능한 스케일과 이동값을 곱하고 더해줍니다.

```
LayerNorm(x) = γ · (x - 평균) / sqrt(분산 + ε) + β
```

여기서 γ(감마)와 β(베타)는 학습되는 파라미터로, 정규화 이후 모델이 필요한 만큼 다시 스케일을 조정할 수 있게 해줍니다.

## 피드포워드 네트워크: 토큰별 비선형 변환

어텐션이 "토큰들 사이의 관계"를 계산하는 역할이라면, 피드포워드 네트워크는 각 토큰의 벡터를 독립적으로 더 깊이 있게 가공하는 역할을 합니다. 구조는 단순합니다: 차원을 확 늘렸다가(보통 4배) 다시 원래 차원으로 줄이는 2개의 선형층 사이에 비선형 활성화 함수(GELU)를 넣습니다.

```
FFN(x) = Linear2( GELU( Linear1(x) ) )
```

차원을 넓혔다가 좁히는 이유는, 중간에 더 넓은 공간에서 다양한 특징을 조합해볼 여유를 주기 위함입니다.

## Pre-LN 구조

LayerNorm을 어디에 놓을지에 따라 Post-LN과 Pre-LN 두 방식이 있습니다. 원조 트랜스포머 논문은 Post-LN(어텐션/FFN 이후에 정규화)을 썼지만, GPT-2 이후 대부분의 모델은 **Pre-LN**(어텐션/FFN 이전에 정규화) 방식을 사용합니다. Pre-LN이 학습 초기에 훨씬 안정적이기 때문입니다. 이 책도 Pre-LN을 사용합니다.

```
x = x + Attention(LayerNorm(x))
x = x + FFN(LayerNorm(x))
```

## 구현하기

In [ ]:
%%writefile mini_gpt/transformer_block.py
"""트랜스포머 블록: Attention + FeedForward + LayerNorm + Residual"""
import torch.nn as nn
from mini_gpt.attention import MultiHeadAttention


class FeedForward(nn.Module):
    def __init__(self, embed_dim, hidden_mult=4, dropout=0.1):
        super().__init__()
        hidden_dim = embed_dim * hidden_mult
        self.net = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, max_seq_len, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_dim)
        self.attention = MultiHeadAttention(embed_dim, num_heads, max_seq_len, dropout)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.feed_forward = FeedForward(embed_dim, hidden_mult=4, dropout=dropout)

    def forward(self, x):
        # Pre-LN 구조: 정규화 -> 서브 레이어 -> 잔차 연결
        x = x + self.attention(self.ln1(x))
        x = x + self.feed_forward(self.ln2(x))
        return x

## 동작 확인

In [ ]:
from mini_gpt.transformer_block import TransformerBlock
import torch

embed_dim = 64
seq_len = 10
batch_size = 2

x = torch.randn(batch_size, seq_len, embed_dim)
block = TransformerBlock(embed_dim=embed_dim, num_heads=4, max_seq_len=32)

output = block(x)
print("입력 shape:", x.shape)
print("출력 shape:", output.shape)  # 입력과 동일해야 함 - 블록을 여러 겹 쌓을 수 있는 이유
print("파라미터 개수:", sum(p.numel() for p in block.parameters()))

블록의 입력과 출력 shape이 완전히 동일하다는 점이 중요합니다. 그래야 이 블록을 레고 블록처럼 원하는 개수만큼 쌓을 수 있습니다.

In [ ]:
# 잔차 연결이 실제로 그래디언트를 안정시키는지 간단히 확인
import torch.nn as nn

# 블록 12개를 깊게 쌓아도 그래디언트가 소실되지 않는지 확인
blocks = nn.ModuleList([
    TransformerBlock(embed_dim=64, num_heads=4, max_seq_len=32) for _ in range(12)
])

x = torch.randn(1, 10, 64, requires_grad=True)
out = x
for b in blocks:
    out = b(out)

loss = out.sum()
loss.backward()

print("입력 x의 그래디언트 절댓값 평균:", x.grad.abs().mean().item())
print("-> 0에 가깝지 않다는 것은 12개 층을 통과해도 그래디언트가 잘 전달된다는 뜻입니다.")

다음 장에서는 이 트랜스포머 블록을 N개 쌓고, 임베딩 층과 출력층을 연결해서 완전한 미니 GPT 모델을 조립합니다.

# 미니 GPT 모델 조립하기

지금까지 만든 부품들을 정리하면 다음과 같습니다.

- `GPTEmbedding`: 토큰 ID → 벡터 (토큰 임베딩 + 위치 임베딩)
- `TransformerBlock`: 어텐션 + 피드포워드 (N개를 쌓을 것)
- 마지막으로 필요한 것: 트랜스포머 블록들의 출력을 다시 **"어휘 사전 크기만큼의 확률 분포"**로 바꿔주는 출력층

이번 장에서는 이 부품들을 조립해 완전한 GPT 모델 클래스를 만듭니다.

## 출력층(LM Head)

트랜스포머 블록을 통과한 후 각 토큰 위치의 벡터는 여전히 `embed_dim` 크기입니다. 이 벡터를 "다음 토큰이 어휘 사전의 각 단어일 확률"로 바꾸려면, `embed_dim` → `vocab_size`로 변환하는 선형층이 필요합니다. 이를 **LM Head(Language Modeling Head)**라고 부릅니다.

```
로짓(logits) = Linear(embed_dim -> vocab_size)(트랜스포머 출력)
```

로짓에 softmax를 적용하면 어휘 사전 전체에 대한 확률 분포가 됩니다. 실제 학습에서는 softmax를 명시적으로 적용하지 않고, `CrossEntropyLoss`가 내부적으로 softmax와 로그를 함께 계산하도록 맡깁니다 (수치적으로 더 안정적입니다).

## 가중치 공유(Weight Tying)

GPT-2를 포함한 많은 모델은 토큰 임베딩 행렬과 LM Head의 가중치를 **공유**합니다. 둘 다 "토큰 ↔ 벡터" 변환을 담당하는 역할이 비슷하기 때문입니다 (하나는 ID→벡터, 하나는 벡터→ID별 점수). 이렇게 하면 파라미터 수를 크게 줄이면서 성능은 거의 유지되거나 오히려 좋아지는 경우가 많습니다. 이 책의 미니 GPT도 이 기법을 사용합니다.

## 구현하기

In [ ]:
%%writefile mini_gpt/model.py
"""미니 GPT: 임베딩 + 트랜스포머 블록 N개 + 출력층"""
import torch
import torch.nn as nn
from mini_gpt.embedding import GPTEmbedding
from mini_gpt.transformer_block import TransformerBlock


class MiniGPT(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, num_heads=4,
                 num_layers=4, max_seq_len=128, dropout=0.1):
        super().__init__()
        self.max_seq_len = max_seq_len

        self.embedding = GPTEmbedding(vocab_size, embed_dim, max_seq_len, dropout)

        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, max_seq_len, dropout)
            for _ in range(num_layers)
        ])

        self.final_ln = nn.LayerNorm(embed_dim)
        self.lm_head = nn.Linear(embed_dim, vocab_size, bias=False)

        # 가중치 공유: 토큰 임베딩과 출력층이 같은 행렬을 사용
        self.lm_head.weight = self.embedding.token_embedding.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):
        # GPT-2와 유사한 초기화: 작은 표준편차의 정규분포
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, token_ids, targets=None):
        # token_ids: (batch, seq_len)
        x = self.embedding(token_ids)          # (batch, seq_len, embed_dim)

        for block in self.blocks:
            x = block(x)

        x = self.final_ln(x)
        logits = self.lm_head(x)               # (batch, seq_len, vocab_size)

        loss = None
        if targets is not None:
            # logits: (batch, seq_len, vocab_size) -> (batch*seq_len, vocab_size)
            # targets: (batch, seq_len) -> (batch*seq_len,)
            loss = nn.functional.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
            )

        return logits, loss

    def num_parameters(self):
        return sum(p.numel() for p in self.parameters())

## 모델 생성 및 파라미터 수 확인

In [ ]:
from mini_gpt.model import MiniGPT
import torch

model = MiniGPT(
    vocab_size=500,     # 2장 토크나이저의 어휘 크기
    embed_dim=128,
    num_heads=4,
    num_layers=4,
    max_seq_len=64,
    dropout=0.1,
)

print(model)
print("\n총 파라미터 수:", f"{model.num_parameters():,}")

## 순전파(Forward Pass) 확인

In [ ]:
# 가짜 배치로 순전파가 정상 동작하는지 확인
batch_size = 4
seq_len = 16

fake_input = torch.randint(0, 500, (batch_size, seq_len))
fake_target = torch.randint(0, 500, (batch_size, seq_len))

logits, loss = model(fake_input, targets=fake_target)

print("입력 shape:", fake_input.shape)
print("로짓 shape:", logits.shape)        # (4, 16, 500)
print("손실값(loss):", loss.item())
print("\n손실값 해석: 학습 전 무작위 초기화 상태이므로 -log(1/500) =",
      round(torch.log(torch.tensor(500.0)).item(), 3), "근처가 나오는 것이 정상입니다.")

손실값이 `log(vocab_size)` 근처로 나온다면 모델이 정상적으로 초기화되었다는 신호입니다. 이는 학습 전 모델이 모든 토큰에 대해 균등한 확률(1/500)을 예측하고 있다는 뜻이기 때문입니다. 만약 이 값이 훨씬 크거나 `nan`이 나온다면 초기화나 구현에 버그가 있다는 신호이므로, 여기서 확인하고 넘어가는 것이 중요합니다.

## 모델 규모를 바꿔가며 실험하기

In [ ]:
# 하이퍼파라미터를 바꿔가며 파라미터 수가 어떻게 변하는지 비교
configs = [
    {"embed_dim": 64,  "num_heads": 2, "num_layers": 2},
    {"embed_dim": 128, "num_heads": 4, "num_layers": 4},
    {"embed_dim": 256, "num_heads": 8, "num_layers": 6},
]

for cfg in configs:
    m = MiniGPT(vocab_size=500, max_seq_len=64, **cfg)
    print(cfg, "-> 파라미터 수:", f"{m.num_parameters():,}")

`embed_dim`이나 `num_layers`를 키우면 파라미터 수가 어떻게 늘어나는지 직접 관찰해보세요. 이 관계를 이해하면 실제 GPT-3(1750억), GPT-4 같은 모델들이 왜 그렇게 많은 파라미터를 갖는지(층을 매우 깊게, 차원을 매우 넓게 쌓았기 때문) 감을 잡을 수 있습니다.

다음 장에서는 이 모델을 실제 텍스트 데이터로 학습시키는 학습 루프를 구현합니다.

# 학습 루프 구현하기

이제 미니 GPT를 실제 텍스트로 학습시켜보겠습니다. 학습 루프의 핵심 아이디어는 간단합니다: **어떤 텍스트의 한 조각을 모델에 주고, 각 위치에서 "바로 다음 토큰"을 맞히게 시킨 뒤, 틀린 만큼 가중치를 조금씩 수정**하는 것을 수백~수천 번 반복합니다.

## 학습 데이터 준비: 입력과 정답을 한 칸씩 밀기

언어모델 학습에서 정답 레이블은 별도로 사람이 만들 필요가 없습니다. 원본 텍스트 자체가 곧 정답입니다. 예를 들어 토큰 시퀀스가 `[나, 는, 밥, 을, 먹, 었, 다]`라면:

- 입력: `[나, 는, 밥, 을, 먹, 었]`
- 정답: `[는, 밥, 을, 먹, 었, 다]` (입력을 한 칸 왼쪽으로 민 것)

즉 입력의 i번째 위치에서 모델은 정답의 i번째 토큰(=입력의 i+1번째 토큰)을 맞혀야 합니다. 이런 방식으로 레이블을 별도로 만들지 않고 원본 텍스트만으로 대량의 학습 데이터를 만들 수 있다는 것이 언어모델 사전학습의 큰 장점입니다 (이를 자기지도학습, self-supervised learning이라 부릅니다).

## 데이터셋 클래스 구현

In [ ]:
%%writefile mini_gpt/dataset.py
"""텍스트 -> (입력, 정답) 학습 배치를 만드는 데이터셋"""
import torch
from torch.utils.data import Dataset


class TextDataset(Dataset):
    def __init__(self, token_ids, seq_len):
        """token_ids: 전체 텍스트를 토크나이징한 긴 정수 리스트"""
        self.token_ids = torch.tensor(token_ids, dtype=torch.long)
        self.seq_len = seq_len

    def __len__(self):
        # seq_len+1 크기의 조각을 겹치지 않게 뽑을 수 있는 시작 위치의 개수
        return len(self.token_ids) - self.seq_len

    def __getitem__(self, idx):
        chunk = self.token_ids[idx: idx + self.seq_len + 1]
        x = chunk[:-1]   # 입력: 앞의 seq_len개
        y = chunk[1:]    # 정답: 한 칸씩 민 seq_len개
        return x, y

## 학습 데이터 준비 (Colab에서 실행)

이번에는 위키백과 한국어 문서 일부를 예시 데이터로 사용합니다. 실습용이므로 짧은 공개 텍스트를 사용하지만, 여러분의 프로젝트에서는 원하는 텍스트 파일로 바꿔도 됩니다.

In [ ]:
from datasets import load_dataset

# 실습용으로 위키텍스트의 아주 작은 영어 서브셋 사용 (다운로드가 빠름)
raw_dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train[:2%]")
text_corpus = "\n".join([t for t in raw_dataset["text"] if len(t.strip()) > 0])
print("전체 글자 수:", len(text_corpus))
print("샘플:\n", text_corpus[:500])

In [ ]:
# 2장에서 만든 BPE 토크나이저를 이 코퍼스로 학습
from mini_gpt.tokenizer import BPETokenizer

tokenizer = BPETokenizer()
tokenizer.train(text_corpus, vocab_size=2000)

all_token_ids = tokenizer.encode(text_corpus)
print("전체 토큰 수:", len(all_token_ids))
print("어휘 사전 크기:", len(tokenizer.vocab))

In [ ]:
from mini_gpt.dataset import TextDataset
from torch.utils.data import DataLoader

seq_len = 64
dataset = TextDataset(all_token_ids, seq_len=seq_len)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

x_batch, y_batch = next(iter(dataloader))
print("입력 배치 shape:", x_batch.shape)   # (32, 64)
print("정답 배치 shape:", y_batch.shape)   # (32, 64)

## 학습 루프

학습의 매 단계(step)는 다음 5단계로 이루어집니다.

1. **순전파(forward)**: 모델에 입력을 넣어 예측(로짓)과 손실(loss)을 계산
2. **역전파(backward)**: 손실을 기준으로 각 파라미터가 손실에 얼마나 기여했는지(그래디언트) 계산
3. **그래디언트 초기화**: 이전 스텝의 그래디언트가 누적되지 않도록 0으로 초기화
4. **옵티마이저 스텝**: 계산된 그래디언트 방향으로 파라미터를 조금씩 이동
5. 다음 배치로 반복

In [ ]:
%%writefile mini_gpt/train.py
"""학습 루프"""
import torch


def train_model(model, dataloader, num_epochs, learning_rate, device):
    model.to(device)
    model.train()

    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    # 학습 후반부로 갈수록 학습률을 서서히 줄여 더 안정적으로 수렴하게 함
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=num_epochs * len(dataloader)
    )

    history = []
    step = 0
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        for x_batch, y_batch in dataloader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)

            logits, loss = model(x_batch, targets=y_batch)

            optimizer.zero_grad()
            loss.backward()
            # 그래디언트 폭발 방지: 그래디언트 크기를 일정 범위로 제한
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()

            epoch_loss += loss.item()
            step += 1
            history.append(loss.item())

            if step % 50 == 0:
                print(f"epoch {epoch+1} | step {step} | loss {loss.item():.4f}")

        avg_loss = epoch_loss / len(dataloader)
        print(f"=== epoch {epoch+1} 평균 loss: {avg_loss:.4f} ===")

    return history

## 실제로 학습시켜보기

In [ ]:
from mini_gpt.model import MiniGPT
from mini_gpt.train import train_model
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

model = MiniGPT(
    vocab_size=len(tokenizer.vocab),
    embed_dim=128,
    num_heads=4,
    num_layers=4,
    max_seq_len=seq_len,
    dropout=0.1,
)
print("파라미터 수:", f"{model.num_parameters():,}")

history = train_model(
    model,
    dataloader,
    num_epochs=3,
    learning_rate=3e-4,
    device=device,
)

## 학습 곡선 확인하기

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(history)
plt.xlabel("학습 스텝")
plt.ylabel("Loss")
plt.title("학습 손실 추이")
plt.grid(True, alpha=0.3)
plt.show()

Loss가 꾸준히 우하향한다면 모델이 데이터의 패턴(자주 등장하는 단어 조합, 문법 구조)을 학습하고 있다는 뜻입니다. 처음 loss는 `log(vocab_size)` 근처에서 시작해서, 학습이 진행될수록 점점 낮아지는 것이 정상입니다.

## 체크포인트 저장하기

In [ ]:
import pickle

torch.save(model.state_dict(), "mini_gpt_checkpoint.pt")
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

print("모델과 토크나이저를 저장했습니다.")

Colab 세션은 일정 시간이 지나면 초기화되므로, 중요한 체크포인트는 `from google.colab import drive; drive.mount('/content/drive')`로 구글 드라이브에 마운트한 뒤 그 경로에 저장해두는 것을 권장합니다.

다음 장에서는 학습된 모델로 실제 텍스트를 생성하는 방법과, 생성 결과의 품질을 좌우하는 샘플링 전략들을 다룹니다.

# 텍스트 생성과 샘플링 전략

## 생성은 "다음 토큰 예측"의 반복

학습된 모델은 주어진 토큰 시퀀스에 대해 "다음 토큰이 무엇일지"에 대한 확률 분포(로짓)를 출력합니다. 텍스트 생성은 이 과정을 반복하는 것입니다.

1. 현재까지의 토큰 시퀀스를 모델에 넣는다.
2. 마지막 위치의 로짓을 확률 분포로 바꾼다.
3. 그 분포에서 다음 토큰 하나를 선택한다.
4. 선택한 토큰을 시퀀스 끝에 이어붙인다.
5. 원하는 길이가 될 때까지 1~4를 반복한다.

여기서 3번, "분포에서 다음 토큰을 어떻게 선택하는가"가 생성된 글의 품질과 창의성을 크게 좌우합니다. 같은 모델이라도 샘플링 전략에 따라 전혀 다른 느낌의 글이 나옵니다.

## 샘플링 전략들

### Greedy Decoding (탐욕적 선택)
매번 확률이 가장 높은 토큰만 선택합니다. 가장 "안전한" 문장이 나오지만, 반복적이고 지루한 글이 되기 쉽습니다.

### Temperature (온도)
softmax를 적용하기 전에 로짓을 온도 값으로 나눕니다.

```
확률 = softmax(로짓 / temperature)
```

- `temperature < 1`: 확률 분포가 더 뾰족해짐 → 확신에 찬(하지만 단조로운) 출력
- `temperature > 1`: 확률 분포가 더 평평해짐 → 다양하지만 엉뚱해질 위험 증가
- `temperature = 1`: 모델이 학습한 원래 분포 그대로 사용

### Top-k 샘플링
확률이 가장 높은 상위 k개의 토큰만 후보로 남기고, 나머지는 확률을 0으로 만든 뒤 그 안에서 샘플링합니다. 말이 안 되는 토큰이 낮은 확률로라도 뽑히는 것을 방지합니다.

### Top-p (Nucleus) 샘플링
누적 확률이 p(예: 0.9)를 넘을 때까지 확률이 높은 순서로 토큰을 후보에 추가하고, 그 안에서 샘플링합니다. Top-k는 후보 개수를 고정하지만, Top-p는 분포가 뾰족할 때는 후보를 적게, 평평할 때는 후보를 많이 남기는 식으로 상황에 맞게 유동적으로 조절됩니다. 실제 서비스에서 가장 널리 쓰이는 방식입니다.

## 구현하기

In [ ]:
%%writefile mini_gpt/generate.py
"""텍스트 생성: greedy, temperature, top-k, top-p 샘플링"""
import torch
import torch.nn.functional as F


@torch.no_grad()
def generate(model, tokenizer, prompt, max_new_tokens=50,
             temperature=1.0, top_k=None, top_p=None, device="cpu"):
    model.eval()
    model.to(device)

    token_ids = tokenizer.encode(prompt)
    token_ids = torch.tensor(token_ids, dtype=torch.long, device=device).unsqueeze(0)

    for _ in range(max_new_tokens):
        # 모델이 학습할 때 본 최대 길이를 넘지 않도록 최근 토큰만 사용
        context = token_ids[:, -model.max_seq_len:]

        logits, _ = model(context)
        last_logits = logits[:, -1, :]  # 마지막 위치의 로짓: (batch, vocab_size)

        # temperature 적용
        last_logits = last_logits / max(temperature, 1e-5)

        # top-k 필터링: 상위 k개 외에는 확률을 -inf로
        if top_k is not None:
            topk_values, _ = torch.topk(last_logits, top_k)
            threshold = topk_values[:, -1].unsqueeze(-1)
            last_logits = torch.where(
                last_logits < threshold,
                torch.full_like(last_logits, float("-inf")),
                last_logits,
            )

        # top-p(nucleus) 필터링
        if top_p is not None:
            sorted_logits, sorted_idx = torch.sort(last_logits, descending=True)
            sorted_probs = F.softmax(sorted_logits, dim=-1)
            cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

            # 누적 확률이 top_p를 넘어서는 지점부터는 제거 대상으로 표시
            remove_mask = cumulative_probs > top_p
            # 첫 번째 토큰은 항상 남겨서 후보가 아예 없는 경우를 방지
            remove_mask[:, 1:] = remove_mask[:, :-1].clone()
            remove_mask[:, 0] = False

            sorted_logits[remove_mask] = float("-inf")
            last_logits = torch.full_like(last_logits, float("-inf"))
            last_logits.scatter_(1, sorted_idx, sorted_logits)

        probs = F.softmax(last_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)  # 확률적 샘플링

        token_ids = torch.cat([token_ids, next_token], dim=1)

    generated_ids = token_ids[0].tolist()
    return tokenizer.decode(generated_ids)

## 실제로 생성해보기

In [ ]:
from mini_gpt.generate import generate

prompt = "The history of"

print("=== Greedy (temperature=0에 가깝게) ===")
print(generate(model, tokenizer, prompt, max_new_tokens=40,
                temperature=0.01, device=device))

print("\n=== Temperature 1.0 ===")
print(generate(model, tokenizer, prompt, max_new_tokens=40,
                temperature=1.0, device=device))

print("\n=== Top-k 샘플링 (k=20) ===")
print(generate(model, tokenizer, prompt, max_new_tokens=40,
                temperature=1.0, top_k=20, device=device))

print("\n=== Top-p 샘플링 (p=0.9) ===")
print(generate(model, tokenizer, prompt, max_new_tokens=40,
                temperature=0.8, top_p=0.9, device=device))

이 책의 미니 GPT는 학습 데이터와 파라미터 수가 매우 작기 때문에, 문법적으로 완벽하거나 사실에 부합하는 문장을 기대하기는 어렵습니다. 하지만 학습이 잘 진행되었다면 영어 단어의 형태, 흔한 조합, 문장 부호 사용 패턴 정도는 그럴듯하게 흉내 내는 것을 관찰할 수 있습니다. 이 실습의 목적은 "실제 상용급 모델을 만드는 것"이 아니라, **동일한 원리가 어떻게 훨씬 큰 규모로 확장되면 ChatGPT 같은 모델이 되는지 체험**하는 데 있습니다.

## 파라미터별 체감 비교

In [ ]:
# 같은 프롬프트, 다른 temperature로 결과가 얼마나 달라지는지 비교
for temp in [0.3, 0.7, 1.2]:
    print(f"\n--- temperature={temp} ---")
    for _ in range(2):
        print(generate(model, tokenizer, prompt, max_new_tokens=30,
                        temperature=temp, top_p=0.9, device=device))

temperature가 낮을수록 두 번 생성한 결과가 서로 비슷해지고, 높을수록 매번 더 다양하고 예측 불가능한 결과가 나오는 것을 확인할 수 있습니다.

다음 장에서는 우리가 처음부터 만든 이 구조가, 실제로 수십억 개의 파라미터를 가진 사전학습 모델과 원리적으로 동일하다는 것을 Hugging Face 라이브러리를 통해 확인해봅니다.

# Hugging Face로 진짜 사전학습 모델 다뤄보기

지금까지 만든 미니 GPT는 파라미터 수가 수십만~수백만 개 수준이고, 학습 데이터도 매우 작았습니다. 이번 장에서는 우리가 만든 것과 **구조적으로 동일하지만** 훨씬 큰 규모로, 훨씬 많은 데이터로 학습된 실제 공개 모델을 Hugging Face `transformers` 라이브러리로 불러와봅니다. 직접 만든 구조와 비교해보면 "규모만 다를 뿐 원리는 같다"는 것을 체감할 수 있습니다.

## transformers 라이브러리 기본 사용법

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "gpt2"  # 1억 2천만 파라미터의 소형 공개 모델

hf_tokenizer = AutoTokenizer.from_pretrained(model_name)
hf_model = AutoModelForCausalLM.from_pretrained(model_name)
hf_model.eval()

print("파라미터 수:", f"{sum(p.numel() for p in hf_model.parameters()):,}")
print(hf_model.config)

`hf_model.config`를 출력해보면 `n_embd`(임베딩 차원), `n_layer`(레이어 수), `n_head`(헤드 수) 같은 값이 나오는데, 이는 우리가 `MiniGPT`를 만들 때 사용한 `embed_dim`, `num_layers`, `num_heads`와 정확히 대응됩니다. GPT-2는 이 값들이 각각 768, 12, 12로 우리 미니 GPT보다 훨씬 큽니다.

## 사전학습 모델로 텍스트 생성

In [ ]:
prompt = "The key idea behind the Transformer architecture is"
inputs = hf_tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    output_ids = hf_model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.8,
        top_p=0.9,
        top_k=50,
        pad_token_id=hf_tokenizer.eos_token_id,
    )

print(hf_tokenizer.decode(output_ids[0], skip_special_tokens=True))

`generate()` 함수의 `temperature`, `top_p`, `top_k` 인자가 8장에서 직접 구현했던 것과 이름까지 동일합니다. 우리가 만든 `generate()` 함수가 바로 이 함수의 축소판이었던 셈입니다.

## 모델 내부 구조 직접 들여다보기

In [ ]:
# GPT-2 내부의 트랜스포머 블록 목록 확인
print(hf_model.transformer.h)  # h = "hidden layers", 우리의 self.blocks와 동일한 역할

print("\n블록 개수:", len(hf_model.transformer.h))
print("첫 번째 블록 구조:")
print(hf_model.transformer.h[0])

출력을 보면 `attn`(어텐션), `mlp`(우리의 FeedForward), `ln_1`, `ln_2`(우리의 ln1, ln2)가 그대로 보입니다. 변수 이름과 세부 구현은 조금 다르지만, 5장에서 만든 `TransformerBlock`과 완전히 같은 구성 요소로 이루어져 있습니다.

## 어텐션 가중치 직접 뽑아서 시각화

In [ ]:
import matplotlib.pyplot as plt

inputs = hf_tokenizer("The cat sat on the mat because it was tired", return_tensors="pt")
with torch.no_grad():
    outputs = hf_model(**inputs, output_attentions=True)

# outputs.attentions: 레이어별 (batch, num_heads, seq_len, seq_len) 텐서 튜플
first_layer_attn = outputs.attentions[0][0]  # (num_heads, seq_len, seq_len)
tokens = hf_tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for i, ax in enumerate(axes):
    ax.imshow(first_layer_attn[i].numpy(), cmap="viridis")
    ax.set_xticks(range(len(tokens)))
    ax.set_yticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=90, fontsize=8)
    ax.set_yticklabels(tokens, fontsize=8)
    ax.set_title(f"Head {i}")
plt.tight_layout()
plt.show()

각 헤드가 서로 다른 패턴으로 단어들을 연결하는 것을 볼 수 있습니다. 예를 들어 어떤 헤드는 "it"이 "cat"을 가리키는 대명사 관계에 강하게 반응하기도 합니다. 이것이 4장에서 설명한 "Multi-Head Attention이 여러 관점을 동시에 포착한다"는 말의 실제 모습입니다.

## 우리 모델과 GPT-2의 차이 요약

| 항목 | 이 책의 MiniGPT | GPT-2 (small) |
|---|---|---|
| 임베딩 차원 | 64~128 | 768 |
| 레이어 수 | 4~6 | 12 |
| 헤드 수 | 4 | 12 |
| 파라미터 수 | 수십만~수백만 | 1.24억 |
| 학습 토큰 수 | 수만 | 약 100억 |
| 어휘 사전 크기 | 500~2000 | 50,257 |

표에서 알 수 있듯, 구조의 "종류"는 완전히 같고 "규모"만 다릅니다. 이것이 LLM 스케일링(scaling)의 핵심입니다: 같은 구조를 유지한 채 데이터, 파라미터, 연산량을 키우면 성능이 예측 가능한 방식으로 향상된다는 것이 최근 몇 년간 LLM 연구에서 반복적으로 확인된 경험적 법칙(스케일링 법칙, scaling law)입니다.

다음 장에서는 이런 사전학습 모델을 처음부터 다시 학습시키지 않고, 적은 자원으로 특정 목적에 맞게 조정하는 파인튜닝, 특히 **LoRA**라는 경량화 기법을 다룹니다.

# LoRA로 가볍게 파인튜닝하기

## 왜 전체 파인튜닝은 부담스러운가

사전학습된 모델을 특정 작업(예: 특정 말투로 대답하기, 특정 도메인 지식 반영하기)에 맞추려면 추가 학습이 필요합니다. 가장 직관적인 방법은 **모델의 모든 파라미터**를 다시 학습시키는 것이지만, 이는 몇 가지 현실적인 문제가 있습니다.

- 모델이 클수록 모든 파라미터의 그래디언트와 옵티마이저 상태를 GPU 메모리에 올려야 해서 메모리 사용량이 매우 큽니다.
- 파인튜닝 결과물마다 원본 모델 전체 크기만큼의 저장 공간이 다시 필요합니다.
- 데이터가 적을 때 전체 파라미터를 다 바꾸면 기존에 학습된 지식을 잊어버리는 **파국적 망각(catastrophic forgetting)**이 발생하기 쉽습니다.

## LoRA의 핵심 아이디어

**LoRA(Low-Rank Adaptation)**는 원본 모델의 가중치는 그대로 얼려두고(freeze), 각 레이어 옆에 아주 작은 두 개의 행렬(A, B)을 추가로 붙여서 그 작은 행렬만 학습하는 방법입니다.

원래 가중치 업데이트는 다음과 같이 표현할 수 있습니다.

```
새 가중치 = 원본 가중치 W + ΔW
```

일반적인 파인튜닝은 `ΔW`가 `W`와 같은 크기의 완전한 행렬입니다. LoRA는 `ΔW`를 두 개의 훨씬 작은 행렬의 곱으로 근사합니다.

```
ΔW ≈ B · A     (A: r×d 크기, B: d×r 크기, r은 d보다 훨씬 작은 값, 예: r=8)
```

`r`(rank)이 작을수록 학습해야 할 파라미터 수가 극적으로 줄어듭니다. 예를 들어 `d=768`인 레이어를 통째로 파인튜닝하면 `768×768 ≈ 59만` 개의 파라미터가 필요하지만, `r=8`인 LoRA는 `768×8 + 8×768 ≈ 1.2만` 개, 약 2%만 학습하면 됩니다. 놀랍게도 많은 작업에서 이 방식이 전체 파인튜닝과 비슷한 성능을 냅니다.

이렇게 하면 다음 장점이 생깁니다.

- 학습해야 할 파라미터 수와 GPU 메모리 사용량이 크게 줄어듭니다.
- 원본 가중치는 그대로이므로 LoRA 어댑터(A, B 행렬)만 저장하면 되어 결과물 용량이 수십 MB 수준으로 작아집니다.
- 여러 개의 LoRA 어댑터를 만들어두고 상황에 따라 갈아 끼울 수 있습니다.

## Hugging Face의 peft 라이브러리로 실습하기

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType
import torch

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(model_name)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,                       # LoRA rank: 작을수록 파라미터 수가 적음
    lora_alpha=16,             # 스케일링 계수 (보통 r의 2배)
    lora_dropout=0.05,
    target_modules=["c_attn"],  # GPT-2에서 Q,K,V를 만드는 어텐션 층에만 LoRA 적용
)

peft_model = get_peft_model(base_model, lora_config)
peft_model.print_trainable_parameters()

`print_trainable_parameters()`를 실행하면 전체 파라미터 대비 학습 가능한(trainable) 파라미터의 비율이 출력됩니다. 보통 1% 미만인 것을 확인할 수 있습니다. 이것이 LoRA가 "가벼운" 파인튜닝이라 불리는 이유입니다.

## 간단한 데이터셋으로 파인튜닝하기

여기서는 예시로 특정 말투(간결하고 단정적인 문체)로 답하도록 소량의 예시 데이터를 만들어 학습해봅니다.

In [ ]:
from torch.utils.data import Dataset, DataLoader

training_examples = [
    "Q: What is a transformer?\nA: A transformer is a neural network built entirely on attention.",
    "Q: What is attention?\nA: Attention lets a model weigh how much each token matters to another.",
    "Q: What is a tokenizer?\nA: A tokenizer converts raw text into integer tokens a model can process.",
    "Q: What is fine-tuning?\nA: Fine-tuning adapts a pretrained model to a new task using extra training.",
] * 30  # 소규모 데이터를 반복하여 패턴을 익히게 함


class SimpleTextDataset(Dataset):
    def __init__(self, examples, tokenizer, max_len=64):
        self.encodings = tokenizer(
            examples, truncation=True, padding="max_length",
            max_length=max_len, return_tensors="pt",
        )

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        input_ids = self.encodings["input_ids"][idx]
        attention_mask = self.encodings["attention_mask"][idx]
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": input_ids.clone(),
        }


train_dataset = SimpleTextDataset(training_examples, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
peft_model.to(device)
peft_model.train()

optimizer = torch.optim.AdamW(peft_model.parameters(), lr=1e-4)

for epoch in range(3):
    total_loss = 0.0
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = peft_model(**batch)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"epoch {epoch+1} 평균 loss: {total_loss/len(train_loader):.4f}")

## LoRA 어댑터 저장 및 재사용

In [ ]:
peft_model.save_pretrained("gpt2-lora-adapter")
print("어댑터 저장 완료. 폴더 용량을 확인해보세요 (원본 모델보다 훨씬 작습니다).")

import os
total_size = sum(
    os.path.getsize(os.path.join("gpt2-lora-adapter", f))
    for f in os.listdir("gpt2-lora-adapter")
)
print(f"어댑터 폴더 크기: {total_size / 1024 / 1024:.2f} MB")

In [ ]:
# 저장된 어댑터를 나중에 다시 불러와서 원본 모델에 붙이는 방법
from peft import PeftModel

fresh_base_model = AutoModelForCausalLM.from_pretrained(model_name)
loaded_model = PeftModel.from_pretrained(fresh_base_model, "gpt2-lora-adapter")

loaded_model.eval()
prompt = "Q: What is a transformer?\nA:"
inputs = tokenizer(prompt, return_tensors="pt")
with torch.no_grad():
    output = loaded_model.generate(**inputs, max_new_tokens=30, do_sample=False)
print(tokenizer.decode(output[0], skip_special_tokens=True))

이처럼 LoRA는 원본 모델(GPT-2, LLaMA 계열, Qwen 등 어떤 모델이든)을 그대로 둔 채 작은 어댑터만 학습·저장·교체할 수 있어, 개인 GPU나 Colab 무료 등급처럼 자원이 제한된 환경에서도 실전 파인튜닝을 경험해볼 수 있는 실용적인 방법입니다.

다음 장에서는 모델의 가중치를 전혀 바꾸지 않고도 원하는 답변을 유도하는 **프롬프트 엔지니어링** 기법들을 살펴봅니다.

# 프롬프트 엔지니어링과 추론 시 제어

파인튜닝은 모델의 가중치 자체를 바꾸는 방법이었습니다. 반면 **프롬프트 엔지니어링(Prompt Engineering)**은 가중치를 전혀 건드리지 않고, 입력(프롬프트)을 설계하는 것만으로 모델의 출력을 원하는 방향으로 유도하는 기법입니다. 비용이 거의 들지 않으면서도 효과가 크기 때문에 실무에서 가장 먼저 시도하는 방법입니다.

## Zero-shot vs Few-shot

- **Zero-shot**: 예시 없이 바로 지시만 주는 방식. `"다음 문장을 한국어로 번역하세요: Hello"`
- **Few-shot**: 원하는 입출력 형식의 예시를 몇 개 프롬프트에 포함시키는 방식. 모델은 예시의 패턴을 보고 그 형식을 따라합니다.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()

def generate_text(prompt, max_new_tokens=30, temperature=0.7):
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=True, temperature=temperature, top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)


zero_shot_prompt = "Translate to French: Good morning"
print("=== Zero-shot ===")
print(generate_text(zero_shot_prompt))

few_shot_prompt = (
    "English: Hello -> French: Bonjour\n"
    "English: Thank you -> French: Merci\n"
    "English: Good morning -> French:"
)
print("\n=== Few-shot ===")
print(generate_text(few_shot_prompt))

Few-shot 프롬프트는 모델에게 "지금 무슨 작업을 하는지, 답을 어떤 형식으로 내야 하는지"를 예시로 직접 보여주기 때문에, 특히 작은 모델일수록 zero-shot보다 훨씬 안정적인 결과를 냅니다. 이는 사전학습 단계에서 모델이 방대한 텍스트 속의 패턴 반복을 이미 많이 학습했기 때문에 가능한 현상입니다.

## 역할 부여와 시스템 프롬프트

ChatGPT 같은 대화형 모델은 대화 시작 부분에 "당신은 유능한 코딩 어시스턴트입니다"와 같은 **시스템 프롬프트**를 넣어 모델의 역할과 답변 스타일을 지정합니다. 원리는 few-shot과 비슷합니다. 모델 입력 맨 앞에 맥락을 깔아주면, 이어지는 생성이 그 맥락에 맞춰 이어질 확률이 높아집니다.

In [ ]:
system_context = (
    "You are a concise technical writer. You answer in exactly one short sentence.\n\n"
)
question = "Q: What is gradient descent?\nA:"

print(generate_text(system_context + question, max_new_tokens=25))

## Chain-of-Thought(생각의 사슬) 프롬프팅

복잡한 추론이 필요한 문제는 "바로 답을 말하라"고 하는 것보다 "단계별로 생각해보라"고 유도했을 때 더 정확한 답을 내는 경향이 관찰됩니다. 이를 Chain-of-Thought(CoT) 프롬프팅이라 부릅니다.

In [ ]:
direct_prompt = "Q: If a train travels 60 miles in 1.5 hours, what is its speed? A:"
cot_prompt = (
    "Q: If a train travels 60 miles in 1.5 hours, what is its speed?\n"
    "A: Let's think step by step. Speed equals distance divided by time. "
    "Distance is 60 miles, time is 1.5 hours. So speed ="
)

print("=== 직접 질문 ===")
print(generate_text(direct_prompt, max_new_tokens=20))
print("\n=== 단계별 사고 유도 ===")
print(generate_text(cot_prompt, max_new_tokens=20))

이 책의 미니 GPT나 소형 GPT-2에서는 효과가 제한적으로 보일 수 있지만, 파라미터 규모가 큰 모델일수록 CoT 프롬프팅의 효과가 뚜렷하게 나타난다는 것이 여러 연구에서 보고되었습니다. 이는 모델 규모가 커지면서 나타나는 "창발적 능력(emergent ability)"의 대표적인 예로 언급됩니다.

## 추론 시 파라미터 요약

지금까지 배운 생성 파라미터들을 정리하면 다음과 같습니다.

| 파라미터 | 역할 | 값을 낮추면 | 값을 높이면 |
|---|---|---|---|
| `temperature` | 확률 분포의 날카로움 조절 | 더 결정적, 반복적 | 더 다양, 무작위 |
| `top_k` | 후보 토큰 개수 제한 | 후보가 적어 안전한 선택 | 더 다양한 후보 허용 |
| `top_p` | 누적 확률 기준 후보 제한 | 후보가 적어짐 | 더 넓은 후보 허용 |
| `max_new_tokens` | 생성할 최대 토큰 수 | 짧은 응답 | 긴 응답 (비용/시간 증가) |

실무에서는 정답이 명확한 작업(코드 생성, 사실 기반 질의응답)에는 `temperature`를 낮게, 창의적인 작업(스토리, 브레인스토밍)에는 `temperature`를 상대적으로 높게 설정하는 것이 일반적인 관행입니다.

다음, 마지막 장에서는 이 책 전체를 정리하고, 여기서 더 나아가고 싶은 독자를 위한 다음 학습 경로를 제안합니다.

# 마무리: 전체 정리와 다음 단계

## 우리가 만든 것 되돌아보기

이 책에서는 텍스트 한 줄이 실제로 어떤 과정을 거쳐 다음 단어를 예측하는지, 아래 파이프라인을 모듈 단위로 하나씩 직접 구현했습니다.

```
mini_gpt/
├── tokenizer.py          # 2장: 텍스트 -> 토큰 ID (BPE)
├── embedding.py           # 3장: 토큰 ID -> 벡터 (토큰 임베딩 + 위치 임베딩)
├── attention.py            # 4장: Self-Attention, Multi-Head Attention
├── transformer_block.py    # 5장: Attention + FeedForward + LayerNorm + Residual
├── model.py                 # 6장: 블록 N개를 쌓은 전체 GPT 모델
├── dataset.py                # 7장: 텍스트를 (입력, 정답) 쌍으로 변환
├── train.py                   # 7장: 학습 루프
└── generate.py                 # 8장: 텍스트 생성과 샘플링 전략
```

그리고 9~11장에서는 이 원리가 실제 GPT-2 같은 사전학습 모델에도 동일하게 적용되며, LoRA로 가볍게 파인튜닝하고, 가중치를 건드리지 않고도 프롬프트만으로 모델의 행동을 조절할 수 있음을 확인했습니다.

## 핵심 개념 요약

- **토큰화**: 텍스트를 모델이 처리 가능한 정수 단위로 쪼갠다. BPE는 자주 등장하는 조각을 점점 큰 단위로 병합해 어휘 사전을 구축한다.
- **임베딩**: 토큰 ID를 의미를 담은 벡터로 바꾸고, 순서 정보가 사라지지 않도록 위치 임베딩을 더한다.
- **어텐션**: Query-Key-Value 구조로 각 토큰이 문맥 속 다른 토큰을 얼마나 참고할지 계산한다. Causal mask로 미래 토큰을 가려 "다음 단어 예측"이라는 과제를 성립시킨다.
- **트랜스포머 블록**: 잔차 연결과 LayerNorm으로 학습을 안정시키고, 어텐션(토큰 간 관계)과 FeedForward(토큰별 변환)를 반복해 표현력을 키운다.
- **학습**: 원본 텍스트를 한 칸 밀어 정답으로 삼는 자기지도학습 방식으로, 별도의 사람 라벨링 없이 대량의 데이터를 활용할 수 있다.
- **생성**: 학습된 확률 분포에서 temperature, top-k, top-p로 다음 토큰을 선택하는 방식에 따라 결과물의 성격이 달라진다.
- **파인튜닝과 LoRA**: 전체 재학습 없이 작은 어댑터 행렬만 학습해 적은 자원으로 모델을 특정 목적에 맞게 조정할 수 있다.
- **프롬프트 엔지니어링**: 가중치를 바꾸지 않고 입력 설계만으로 모델의 출력을 유도한다.

## 실제 대규모 LLM과의 차이, 더 알아볼 만한 주제

이 책의 미니 GPT는 원리를 배우기 위해 의도적으로 축소한 모델입니다. 실제 프로덕션 LLM으로 나아가려면 다음 주제들을 추가로 살펴보는 것을 권합니다.

- **효율적인 어텐션**: Flash Attention, Grouped-Query Attention(GQA) 등 긴 문맥을 더 빠르고 메모리 효율적으로 처리하는 기법
- **위치 인코딩의 발전**: RoPE(Rotary Position Embedding) 등 더 긴 문맥 길이로 일반화가 잘 되는 위치 인코딩 방식
- **RLHF와 정렬(Alignment)**: 인간 피드백을 활용한 강화학습으로 모델이 유용하고 안전하게 답하도록 조정하는 과정
- **양자화(Quantization)**: 모델 가중치를 더 적은 비트로 표현해 메모리와 연산량을 줄이는 기법 (INT8, INT4 등)
- **MoE(Mixture of Experts)**: 입력마다 전체 파라미터 중 일부만 활성화해 연산량 대비 더 큰 모델 용량을 갖게 하는 구조
- **RAG(Retrieval-Augmented Generation)**: 외부 지식 베이스를 검색해 모델의 답변에 실시간 정보를 결합하는 기법
- **평가(Evaluation)**: 벤치마크 데이터셋으로 모델의 추론, 지식, 안전성 등을 정량적으로 측정하는 방법론

## 추천 학습 경로

1. 이 책의 `MiniGPT`에서 `embed_dim`, `num_layers`, `num_heads`를 점점 키워보며 파라미터 수와 학습 시간, 생성 품질의 관계를 직접 관찰해보세요.
2. 직접 준비한 한국어 코퍼스(뉴스, 블로그, 소설 등)로 토크나이저와 모델을 다시 학습시켜보세요.
3. Hugging Face의 `Trainer` API로 전체 파인튜닝과 LoRA 파인튜닝의 결과를 비교해보세요.
4. 원조 논문 "Attention Is All You Need"와 GPT-2 논문("Language Models are Unsupervised Multitask Learners")을 읽고, 이 책에서 구현한 코드와 논문의 수식을 하나씩 대응시켜보세요.

## 마치며

LLM은 마법 같아 보이지만, 이 책에서 확인했듯 그 안에는 토큰화, 행렬 곱셈, softmax, 경사하강법처럼 잘 정의된 구성 요소들이 있을 뿐입니다. 규모가 커지면서 놀라운 능력들이 나타나긴 하지만, 그 기초는 여러분이 이 책에서 직접 코드로 눌러본 바로 이 구조입니다. 이 토대 위에서 더 깊은 논문과 오픈소스 코드를 읽어나가면, 이후의 발전된 기법들도 훨씬 수월하게 이해할 수 있을 것입니다.